#  note
- çıkış layerına göre hangi loss function (cross-entropy ne bekler, binary cross entropy nasıl çalışır, ve bunları türevi alınması)
- ve numeric stability (The manual implementation of the log softmax function is typically done to understand the underlying mechanics of the operation and to ensure custom handling of numerical stability issues.) (exp ve log olan yerler hep sorunlu) (overflow ve underflow problemelri var)
- Küçük ama frameworklerde gradientler negatif olarak verilir
- log derken aslında natural logarithma (ln) denilmesi

In [1]:
import torch

In [4]:
y=(torch.randn(10, 5) < 0.5) * 1.0
y

tensor([[1., 1., 0., 1., 1.],
        [1., 0., 1., 0., 1.],
        [1., 1., 1., 1., 1.],
        [0., 1., 0., 1., 1.],
        [0., 1., 1., 1., 0.],
        [0., 1., 1., 1., 0.],
        [1., 1., 0., 1., 1.],
        [0., 1., 1., 1., 1.],
        [1., 1., 1., 1., 0.],
        [0., 1., 1., 1., 1.]])

In [189]:
def mse_loss(y, y_hat):
    """
    Args:
        y: the label tensor of shape (batch_size, linear_2_out_features)
        y_hat: the prediction tensor of shape (batch_size, linear_2_out_features)

    Return:
        J: scalar loss
        dJdy_hat: The gradient tensor of shape (batch_size, linear_2_out_features)
    """
    # The torch.F.mse_loss function's default behavior is to use the reduction mode 'mean.'
    # Therefore, all the code was written with respect to that.
    #scalar loss
    loss_for_each_y_outputs=((y-y_hat)**2)
    loss_for_each_observation=loss_for_each_y_outputs.mean(axis=1)
    J=loss_for_each_observation.mean(axis=0) 

    # the gradient tensor of y_hat
    # (1/(y.shape[0]*y.shape[1])) is added  for the reduction mode 'mean.'
    dJdy_hat = (2*(y-y_hat)*(-1)) * (1/(y.shape[0]*y.shape[1]))

    """
    Bana not !!!!!!!!!!
    note that Deep learning frameworks choose to represent gradients as the negative of the actual derivative of the loss with respect to model parameters (or predictions) primarily for two reasons:

    1. Consistency with Optimization Algorithms: By representing gradients as the negative direction of increase in the loss, Many optimization  algorithms can directly use the calculated gradients to update parameters in the direction of minimizing the loss. It simplifies the mathematics of optimization algorithms.

    2. Gradient Descent Interpretation: The convention aligns with the intuitive idea of "gradient descent." In gradient descent, you move in the direction opposite to the gradient (negative gradient) to reach the minimum of a function. By using negative gradients, deep learning frameworks maintain this consistency with the traditional interpretation of gradient descent.
    
    """
    return J,dJdy_hat
    

In [197]:
torch.manual_seed(0)
loss = mse_loss
input = torch.randn(3, 5)
print("input , ",input)
target = torch.randn(3, 5)
output= loss(input, target)
print("loss, ",output[0])
print("grad, ",output[1])


input ,  tensor([[ 1.5410, -0.2934, -2.1788,  0.5684, -1.0845],
        [-1.3986,  0.4033,  0.8380, -0.7193, -0.4033],
        [-0.5966,  0.1820, -0.8567,  1.1006, -1.0712]])
loss,  tensor(1.6508)
grad,  tensor([[-0.1891, -0.0364,  0.3403, -0.1947, -0.0566],
        [ 0.2359,  0.1404,  0.0136,  0.1992,  0.0794],
        [ 0.2481, -0.1963,  0.0087, -0.1495,  0.0470]])


In [245]:
torch.manual_seed(0)
loss = torch.nn.MSELoss()
input = torch.randn(3, 5, requires_grad=True)
print("input ",input)
target = torch.randn(3, 5)
output = loss(input, target)
print(output)
print(input.grad)
output.backward()
print(input.grad)

input  tensor([[ 1.5410, -0.2934, -2.1788,  0.5684, -1.0845],
        [-1.3986,  0.4033,  0.8380, -0.7193, -0.4033],
        [-0.5966,  0.1820, -0.8567,  1.1006, -1.0712]], requires_grad=True)
tensor(1.6508, grad_fn=<MseLossBackward0>)
None
tensor([[ 0.1891,  0.0364, -0.3403,  0.1947,  0.0566],
        [-0.2359, -0.1404, -0.0136, -0.1992, -0.0794],
        [-0.2481,  0.1963, -0.0087,  0.1495, -0.0470]])


# bce loss 

In [441]:
def bce_loss(y, y_hat):
    """
    Args:
        y_hat: the prediction tensor of shape (batch_size, linear_2_out_features)
        y: the label tensor of shape (batch_size, linear_2_out_features)
        
    Return:
        J: scalar loss
        dJdy_hat: The gradient tensor of shape (batch_size, linear_2_out_features)
    """

    """
    ->>>> ϵ is a small number to prevent the logarithm from blowing up when y is 0 or 1.
  
    The epsilon value is used to prevent taking the logarithm of zero when calculating the binary cross-entropy loss. This prevents the result 
    from becoming negative infinity (-inf), which can disrupt the training process of a neural network due to numerical instability. 
    By ensuring that the values passed to the log function are bounded away from zero, epsilon helps maintain computational stability.

    Without epsilon:

    log(0) will be -inf
    log(1) is 0, which is fine, but if 1 is due to a floating-point error and is not the true value, gradients can be incorrect.
    With epsilon (let's say epsilon is 1e-12):
    
    log(1e-12) will be a very small number instead of -inf.
    log(1 - 1e-12) will also be a very small number, representing a probability very close to 1 without hitting the extremes.

    """
    
    #Small value to ensure numerical stability
    epsilon=1e-12
    # Clamp the input to prevent log(0)
    y_hat_clamped = torch.clamp(y_hat, epsilon, 1 - epsilon)
    #now, it is time to calculate
    loss_for_each_y_outputs = - (y * torch.log(y_hat_clamped) + (1 - y) * torch.log(1 - y_hat_clamped))
    loss_for_each_observation = loss_for_each_y_outputs.mean(axis=1)
    J=loss_for_each_observation.mean(axis=0) 

    # the gradient tensor of y_hat
    # (1/(y.shape[0]*y.shape[1])) is added  for the reduction mode 'mean.'
    
    # Derivation (dJdy_hat) ):
    #--> (-(y/ y_hat_clamped)) + ((1 - y) / (1 - y_hat_clamped)) but;
    #--> since y_hat is output of sigmoid(z), derivation must have applied to sigmoid  

    dJdy_hat = ((-(y/ (y_hat_clamped))) + ((1 - y) / (1 - (y_hat_clamped)))) * (1/(y.shape[0]*y.shape[1])) 
    #dJdy_hat = (y_hat - y) * (1/(y.shape[0]*y.shape[1])) 

    return J,dJdy_hat


In [442]:
torch.manual_seed(0)

m = nn.Sigmoid()
loss = bce_loss
input =  torch.randn(10, 2, requires_grad=True)
target = (torch.randn(10, 2) < 0.5) * 1.0
output = loss(target,m(input))
print("loss, ",output[0] )
print("grad, ",output[1])

loss,  tensor(0.7357, grad_fn=<MeanBackward1>)
grad,  tensor([[-0.2041, -0.2083],
        [ 0.0889, -0.1272],
        [-0.0775, -0.2868],
        [-0.1203, -0.0578],
        [ 0.1298,  0.0927],
        [ 0.2618, -0.0883],
        [-0.0625, -0.0602],
        [-0.0694, -0.1662],
        [-0.0697, -0.0642],
        [-0.0567,  0.1028]], grad_fn=<MulBackward0>)


In [443]:
torch.manual_seed(0)

m = nn.Sigmoid()
loss = nn.BCELoss(reduction="mean")
input =  torch.randn(10, 2, requires_grad=True)
target = (torch.randn(10, 2) < 0.5) * 1.0

output = loss(m(input), target)
print(output)
print(input.grad)
output.backward()
print(input.grad)

tensor(0.7357, grad_fn=<BinaryCrossEntropyBackward0>)
None
tensor([[-0.0378, -0.0380],
        [ 0.0219, -0.0303],
        [-0.0177, -0.0413],
        [-0.0292, -0.0068],
        [ 0.0307,  0.0230],
        [ 0.0405, -0.0217],
        [-0.0100, -0.0085],
        [-0.0140, -0.0350],
        [-0.0141, -0.0111],
        [-0.0059,  0.0257]])


# cross_entropy_loss

In [2]:
import torch
import torch.nn.functional as F

def cross_entropy_loss(y, y_hat):
    """
    Computes the cross entropy loss and gradient.
    
    Args:
        y: the correct class indices tensor of shape (batch_size,)
        y_hat: the prediction tensor of shape (batch_size, num_classes)

    Return:
        J: scalar loss
        dJdy_hat: The gradient tensor of shape (batch_size, num_classes)
    """

    """
    Categorical Cross-Entropy Loss Function, also known as Cross-Entropy Loss, Softmax Loss,log loss , is
    a loss function used in multiclass classiﬁcation model training. It applies the Softmax Activation
    Function to a model's output (logits) before applying the Negative Log-Likelihood function.

    Here's a step-by-step guide to writing a function for cross-entropy loss:

    1. Apply softmax to your neural network's raw outputs to get probabilities.
    2. Calculate the log of those probabilities (because the cross-entropy formula involves the logarithm of probabilities).
    3. Select the log probability of the correct class for each example in the batch.
    4. Calculate the negative average of these log probabilities (the negative sign is because we want to minimize the loss).
    

    """


    # 1.  Apply softmax to your neural network's raw outputs to get probabilities. (y_hat)
    # first, for numerical stability 
    #Before applying the exponential function in the softmax function, the maximum value of the inputs is subtracted from each input. 
    #This operation does not change the output of the softmax function but prevents large values from being exponentiated,
    #which helps in avoiding overflow.   
    max_y_hat = torch.max(y_hat, dim=1, keepdim=True).values
    exps = torch.exp(y_hat - max_y_hat)
    softmax = exps / torch.sum(exps, dim=1, keepdim=True)

    # 2. Calculate the log of those probabilities
    log_probs = torch.log(softmax)

    # 3. Select the log probability of the correct class for each example in the batch.
    batch_size = y.shape[0]
    actual_log_probs = log_probs[range(batch_size), y]

    # 4. Compute the mean of the negative log probabilities (cross-entropy loss)
    J = -actual_log_probs.mean()

    #--------------------------  Compute gradients------------- (mathematic word de)
    # Calculate the gradient of the loss
    # Initialize gradient tensor with probabilities
    dJdy_hat = softmax.clone()
    # Subtract 1 from the true class probabilities (where y_j = 1)
    dJdy_hat[range(batch_size), y] -= 1
    # Average the gradients over the batch
    dJdy_hat /= batch_size
    return J, dJdy_hat



# Example usage:
torch.manual_seed(0)

y = torch.tensor([2, 1, 0])
y_hat = torch.randn(3, 3, requires_grad=True)

# Compute the loss and gradients
J, dJdy_hat = cross_entropy_loss(y, y_hat)

print("Loss:", J.item())
print("Gradients:\n", dJdy_hat)

# Compare to PyTorch's built-in loss
criterion = torch.nn.CrossEntropyLoss()
J_torch = criterion(y_hat, y)

print("PyTorch Loss:", J_torch.item())
J_torch.backward()
print("PyTorch grad:", y_hat.grad)


Loss: 2.2940218448638916
Gradients:
 tensor([[ 0.2815,  0.0450, -0.3265],
        [ 0.2504, -0.2854,  0.0350],
        [-0.2172,  0.1794,  0.0378]], grad_fn=<DivBackward0>)
PyTorch Loss: 2.2940218448638916
PyTorch grad: tensor([[ 0.2815,  0.0450, -0.3265],
        [ 0.2504, -0.2854,  0.0350],
        [-0.2172,  0.1794,  0.0378]])


# functions

In [54]:
from abc import ABC, abstractmethod
import torch

class ActivationFunctionError(Exception):
    """Custom exception for errors related to activation functions."""

class BaseActivationFunction(ABC):
    """
    Abstract base class for activation functions in neural networks.
    """

    def __init__(self, input_tensor: torch.Tensor):
        self.input_tensor = input_tensor
        self.output = None

    @abstractmethod
    def forward(self) -> torch.Tensor:
        pass

    @abstractmethod
    def backward(self) -> torch.Tensor:
        pass

class SigmoidActivationFunction(BaseActivationFunction):
    """
    Sigmoid activation function implementation.
    """

    def forward(self) -> torch.Tensor:
        self.input_tensor = torch.clamp(self.input_tensor, min=-15, max=15)
        self.output = 1 / (1 + torch.exp(-self.input_tensor))
        return self.output

    def backward(self) -> torch.Tensor:
        if self.output is None:
            raise ActivationFunctionError("Forward pass must be called before backward.")
        return self.output * (1 - self.output)

class ReLUActivationFunction(BaseActivationFunction):
    """
    ReLU activation function implementation.
    """

    def forward(self) -> torch.Tensor:
        self.output = torch.maximum(self.input_tensor, torch.tensor(0.0))
        return self.output

    def backward(self) -> torch.Tensor:
        if self.output is None:
            raise ActivationFunctionError("Forward pass must be called before backward.")
        return (self.input_tensor > 0).float()

class IdentityActivationFunction(BaseActivationFunction):
    """
    Identity activation function implementation.
    """

    def forward(self) -> torch.Tensor:
        self.output = self.input_tensor
        return self.output

    def backward(self) -> torch.Tensor:
        if self.output is None:
            raise ActivationFunctionError("Forward pass must be called before backward.")
        return torch.ones_like(self.input_tensor)

In [ ]:
# Example usage:
torch.manual_seed(0)
y_hat = torch.randn(3, 3, requires_grad=True)
own_activation_func=SigmoidActivationFunction(y_hat)
#own_activation_func=ReLUActivationFunction(y_hat)
#own_activation_func=IdentityActivationFunction(y_hat)
print(own_activation_func.forward())
print(own_activation_func.backward())

In [84]:
torch.manual_seed(0)
y_hat = torch.randn(3, 3, requires_grad=True)
original_func=torch.sigmoid
#original_func=torch.relu
#original_func=torch.Identity
output=original_func(y_hat)
output.backward(torch.ones_like(output))
print(output)
print(y_hat.grad)

tensor([[0.8236, 0.4272, 0.1017],
        [0.6384, 0.2527, 0.1980],
        [0.5995, 0.6980, 0.3276]], grad_fn=<SigmoidBackward0>)
tensor([[0.1453, 0.2447, 0.0913],
        [0.2308, 0.1888, 0.1588],
        [0.2401, 0.2108, 0.2203]])
